In [2]:
import pandas as pd
import plotly.express as pxa
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
raw_df = pd.read_csv('/content/owid-co2-data (1).csv1+++5')

# Filter for years 2000 to 2022 inclusive
df = raw_df[(raw_df['year'] >= 2000) & (raw_df['year'] <= 2022)].copy()

print(f"Filtered: {len(df)} rows | Countries: {df['country'].nunique()} | Years: {df['year'].min()}-{df['year'].max()}")
display(df.head())

Filtered: 5842 rows | Countries: 254 | Years: 2000-2022


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
250,Afghanistan,2000,AFG,20130334.0,1.128379e+10,0.010216,0.000507,1.047128,-0.044512,-4.077542,...,NaN,0.078173,0.000301,0.000505,0.000902,0.000096,16.008562,2.602797,NaN,NaN
251,Afghanistan,2001,AFG,20284303.0,1.102127e+10,0.006538,0.000322,1.069098,0.021970,2.098131,...,NaN,0.077151,0.000300,0.000506,0.000904,0.000097,14.894542,2.701424,NaN,NaN
252,Afghanistan,2002,AFG,21378123.0,1.880487e+10,0.011033,0.000516,1.341065,0.271967,25.438917,...,NaN,0.076608,0.000303,0.000509,0.000911,0.000099,20.036617,3.044495,NaN,NaN
253,Afghanistan,2003,AFG,22733053.0,2.107434e+10,0.009807,0.000431,1.559679,0.218614,16.301525,...,NaN,0.076126,0.000307,0.000512,0.000920,0.000100,20.768833,3.349439,NaN,NaN
254,Afghanistan,2004,AFG,23560656.0,2.233257e+10,0.009807,0.000416,1.237247,-0.322432,-20.672972,...,NaN,0.075725,0.000313,0.000515,0.000930,0.000102,18.933540,3.085296,NaN,NaN


In [3]:
# Format and display the filtered CO2 dataset
print(f"Loaded: {len(df)} rows | Countries: {df['country'].nunique()} | Years: {df['year'].min()}-{df['year'].max()}")
display(df[['country', 'year', 'co2', 'co2_per_capita', 'population', 'gdp']].head())

Loaded: 5842 rows | Countries: 254 | Years: 2000-2022


,country,year,co2,co2_per_capita,population,gdp
250,Afghanistan,2000,1.047128,0.052017,20130334.0,1.128379e+10
251,Afghanistan,2001,1.069098,0.052706,20284303.0,1.102127e+10
252,Afghanistan,2002,1.341065,0.062731,21378123.0,1.880487e+10
253,Afghanistan,2003,1.559679,0.068608,22733053.0,2.107434e+10
254,Afghanistan,2004,1.237247,0.052513,23560656.0,2.233257e+10


In [4]:
print("Countries:", df['country'].unique())
print("\nCO2 range:", df['co2'].min(), "to", df['co2'].max(), "Mt")
print("\nTop emitting countries/regions (2022):")
print(df[df['year']==2022].groupby('country')['co2'].mean().sort_values(ascending=False).head(10).round(1))

Countries: ['Afghanistan' 'Africa' 'Africa (GCP)' 'Albania' 'Algeria' 'Andorra'
 'Angola' 'Anguilla' 'Antarctica' 'Antigua and Barbuda' 'Argentina'
 'Armenia' 'Aruba' 'Asia' 'Asia (GCP)' 'Asia (excl. China and India)'
 'Australia' 'Austria' 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh'
 'Barbados' 'Belarus' 'Belgium' 'Belize' 'Benin' 'Bermuda' 'Bhutan'
 'Bolivia' 'Bonaire Sint Eustatius and Saba' 'Bosnia and Herzegovina'
 'Botswana' 'Brazil' 'British Virgin Islands' 'Brunei' 'Bulgaria'
 'Burkina Faso' 'Burundi' 'Cambodia' 'Cameroon' 'Canada' 'Cape Verde'
 'Central African Republic' 'Central America (GCP)' 'Chad' 'Chile' 'China'
 'Christmas Island' 'Colombia' 'Comoros' 'Congo' 'Cook Islands'
 'Costa Rica' "Cote d'Ivoire" 'Croatia' 'Cuba' 'Curacao' 'Cyprus'
 'Czechia' 'Democratic Republic of Congo' 'Denmark' 'Djibouti' 'Dominica'
 'Dominican Republic' 'East Timor' 'Ecuador' 'Egypt' 'El Salvador'
 'Equatorial Guinea' 'Eritrea' 'Estonia' 'Eswatini' 'Ethiopia' 'Europe'
 'Europe (GCP)' 'Euro

In [5]:
import plotly.graph_objects as go

# 1. Filter out regional aggregates (valid countries typically have a 3-letter ISO code and are not regions)
# We'll filter for rows with valid 3-letter country ISO codes (excluding aggregate regions)
countries_df = df[df['iso_code'].notna() & (df['iso_code'].str.len() == 3)].copy()

# Define target highlight country
target_country = highlight if 'highlight' in globals() else 'India'

# Create a blank figure
fig = go.Figure()

# 2. Plot background lines for all countries in light grey
for country in countries_df['country'].unique():
    if country == target_country:
        continue
    country_data = countries_df[countries_df['country'] == country]
    fig.add_trace(
        go.Scatter(
            x=country_data['year'],
            y=country_data['co2_per_capita'],
            mode='lines',
            line=dict(color='#EEEEEE', width=1),
            name=country,
            hoverinfo='name+x+y',
            showlegend=False
        )
    )

# 3. Plot the highlighted country in blue and thicker line
highlight_data = countries_df[countries_df['country'] == target_country]
fig.add_trace(
    go.Scatter(
        x=highlight_data['year'],
        y=highlight_data['co2_per_capita'],
        mode='lines+markers',
        line=dict(color='#1F77B4', width=3),
        marker=dict(size=5),
        name=target_country,
        hoverinfo='name+x+y',
        showlegend=False
    )
)

# 4. Add direct text label at the end of the highlighted line in blue
last_row = highlight_data.dropna(subset=['co2_per_capita']).iloc[-1]
fig.add_trace(
    go.Scatter(
        x=[last_row['year']],
        y=[last_row['co2_per_capita']],
        mode='markers+text',
        text=[f"  {target_country}"],
        textposition="middle right",
        textfont=dict(color='#1F77B4', size=13, family="Arial-Bold"),
        marker=dict(color='#1F77B4', size=8),
        showlegend=False,
        hoverinfo='skip'
    )
)

# 5. Define layout with an insight title
fig.update_layout(
    title={
        'text': f"<b>India's Steady Rise</b>: CO₂ per Capita Remains Low Globally Despite Sustained Growth (2000-2022)",
        'y': 0.95,
        'x': 0.05,
        'xanchor': 'left',
        'yanchor': 'top',
        'font': dict(size=16)
    },
    xaxis_title="Year",
    yaxis_title="CO₂ per Capita (tonnes)",
    template="plotly_white",
    margin=dict(r=100, t=80, l=50, b=50),
    xaxis=dict(tickmode='linear', tick0=2000, dtick=2),
    height=600
)

fig.show()

In [6]:
import plotly.graph_objects as go

# 1. Identify Asian countries
asian_countries = [
    'Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Brunei', 'Cambodia',
    'China', 'Cyprus', 'Georgia', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan',
    'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Laos', 'Lebanon', 'Malaysia', 'Maldives', 'Mongolia',
    'Myanmar', 'Nepal', 'North Korea', 'Oman', 'Pakistan', 'Palestine', 'Philippines', 'Qatar',
    'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria', 'Taiwan', 'Tajikistan',
    'Thailand', 'East Timor', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan',
    'Vietnam', 'Yemen'
]

# Filter our dataset to include only valid individual Asian countries
asian_df = df[(df['country'].isin(asian_countries)) & (df['iso_code'].str.len() == 3)].copy()

# Define target highlight country
target_country = highlight if 'highlight' in globals() else 'India'

# Create figure
fig = go.Figure()

# 2. Plot background lines for all other Asian countries in light grey
for country in asian_df['country'].unique():
    if country == target_country:
        continue
    country_data = asian_df[asian_df['country'] == country]
    fig.add_trace(
        go.Scatter(
            x=country_data['year'],
            y=country_data['co2_per_capita'],
            mode='lines',
            line=dict(color='#EEEEEE', width=1),
            name=country,
            hoverinfo='name+x+y',
            showlegend=False
        )
    )

# 3. Plot the highlighted Asian country in blue
highlight_data = asian_df[asian_df['country'] == target_country]
fig.add_trace(
    go.Scatter(
        x=highlight_data['year'],
        y=highlight_data['co2_per_capita'],
        mode='lines+markers',
        line=dict(color='#1F77B4', width=3),
        marker=dict(size=5),
        name=target_country,
        hoverinfo='name+x+y',
        showlegend=False
    )
)

# 4. Add direct text label at the end of the highlighted line in blue
last_row = highlight_data.dropna(subset=['co2_per_capita']).iloc[-1]
fig.add_trace(
    go.Scatter(
        x=[last_row['year']],
        y=[last_row['co2_per_capita']],
        mode='markers+text',
        text=[f"  {target_country}"],
        textposition="middle right",
        textfont=dict(color='#1F77B4', size=13, family="Arial-Bold"),
        marker=dict(color='#1F77B4', size=8),
        showlegend=False,
        hoverinfo='skip'
    )
)

# 5. Define layout with an insight title
fig.update_layout(
    title={        'text': f"<b>Emissions in Perspective</b>: {target_country}'s Path Compared to Asian Peers (2000-2022)",
        'y': 0.95,
        'x': 0.05,
        'xanchor': 'left',
        'yanchor': 'top',
        'font': dict(size=16)
    },
    xaxis_title="Year",
    yaxis_title="CO₂ per Capita (tonnes)",
    template="plotly_white",
    margin=dict(r=120, t=80, l=50, b=50),
    xaxis=dict(tickmode='linear', tick0=2000, dtick=2),
    height=600
)

fig.show()